In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import HTML
from IPython.display import display

# ============================================================
# NEWTON AND QUASI-NEWTON OPTIMIZATION
# ============================================================

plt.close('all')
plt.ioff()

CONTENT_WIDTH = '900px'

plt.rcParams.update({'font.size':10.5,'axes.titlesize':12,'axes.labelsize':10.5,'xtick.labelsize':9,'ytick.labelsize':9,'legend.fontsize':8.5})

# ============================================================
# STYLE
# ============================================================

display(HTML("""
<style>

.nq-root{
    width:900px;
    max-width:900px;
    font-family:Arial,sans-serif;
}

.nq-header{
    background:linear-gradient(90deg,#1565c0,#1976d2);
    color:white;
    padding:10px 14px;
    border-radius:8px 8px 0 0;
    font-size:19px;
    font-weight:bold;
}

.nq-doc{
    background:#f6f9fd;
    border:1px solid #b9cce5;
    border-top:none;
    padding:9px 13px;
    border-radius:0 0 8px 8px;
    font-size:14.5px;
    line-height:1.45;
    margin-bottom:7px;
}

.nq-box{
    width:100%;
    box-sizing:border-box;
    border:1px solid #b9cce5;
    border-radius:7px;
    padding:8px 11px;
    margin-bottom:7px;
    font-size:14px;
    line-height:1.45;
}

.nq-title{
    font-weight:bold;
    color:#0d47a1;
    font-size:14.5px;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.jp-OutputArea-output,
.jp-OutputArea-child{
    overflow-x:visible !important;
    overflow-y:visible !important;
    max-width:none !important;
}

.jp-OutputArea,
.output_wrapper,
.widget-box,
.jupyter-widgets{
    overflow:visible !important;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

display(HTML("""
<div class="nq-root">

<div class="nq-header">
Newton and Quasi-Newton Optimization
</div>

<div class="nq-doc">

Newton optimization uses both the gradient and the Hessian matrix to construct
the correction

<div style="text-align:center;font-size:15px;margin:5px 0;">
<b>
δx = -H⁻¹ ∇f(x).
</b>
</div>

For a quadratic objective function, the Hessian is constant and Newton can
reach the minimum in a single ideal step.

Quasi-Newton methods avoid the explicit calculation of second-order
derivatives. Instead, an approximation of the inverse Hessian is progressively
updated from changes in position and gradient.

The example below compares Newton with the BFGS quasi-Newton method for the
same two-variable convex function.

</div>

</div>
"""))

# ============================================================
# OBJECTIVE FUNCTION
# ============================================================

def f(x):

    X = x[0]-2.0
    Y = x[1]+1.0

    return X**2+4.0*Y**2+0.5*X*Y

def grad(x):

    X = x[0]-2.0
    Y = x[1]+1.0

    return np.array([2.0*X+0.5*Y,8.0*Y+0.5*X])

Hessian = np.array([[2.0,0.5],[0.5,8.0]])

minimum = np.array([2.0,-1.0])

x0 = np.array([-3.0,3.0])

# ============================================================
# NEWTON PATH
# ============================================================

newton_path = [x0.copy()]

x = x0.copy()

for k in range(5):

    d = -np.linalg.solve(Hessian,grad(x))

    x = x+d

    newton_path.append(x.copy())

    if np.linalg.norm(d) < 1e-10:

        break

newton_path = np.array(newton_path)

# ============================================================
# BFGS PATH
# ============================================================

bfgs_path = [x0.copy()]

x = x0.copy()

S = np.eye(2)

for k in range(20):

    g = grad(x)

    d = -S@g

    alpha_grid = np.linspace(0.0,1.0,2001)

    values = np.array([f(x+alpha*d) for alpha in alpha_grid])

    alpha = alpha_grid[np.argmin(values)]

    delta = alpha*d

    x_new = x+delta

    gamma = grad(x_new)-g

    D = delta@gamma

    if D > 1e-12:

        rho = 1.0/D

        I = np.eye(2)

        S = (I-rho*np.outer(delta,gamma))@S@(I-rho*np.outer(gamma,delta))+rho*np.outer(delta,delta)

    x = x_new

    bfgs_path.append(x.copy())

    if np.linalg.norm(delta) < 1e-7:

        break

bfgs_path = np.array(bfgs_path)

# ============================================================
# RESULT BOX
# ============================================================

display(HTML(f"""
<div class="nq-root">

<div class="nq-box">

<div class="nq-title">Optimization result</div>

Starting point:
<b>x₀ = ({x0[0]:.1f}, {x0[1]:.1f})</b>

&nbsp;&nbsp;&nbsp;

True minimum:
<b>x* = ({minimum[0]:.1f}, {minimum[1]:.1f})</b>

<br>

Newton iterations:
<b>{len(newton_path)-1}</b>

&nbsp;&nbsp;&nbsp;

BFGS iterations:
<b>{len(bfgs_path)-1}</b>

</div>

</div>
"""))

# ============================================================
# FIGURE
# ============================================================

fig,(ax1,ax2) = plt.subplots(1,2,figsize=(9.0,4.0))

# ============================================================
# CONTOURS AND ITERATION PATHS
# ============================================================

xx = np.linspace(-4,4,250)

yy = np.linspace(-4,4,250)

X,Y = np.meshgrid(xx,yy)

Z = (X-2.0)**2+4.0*(Y+1.0)**2+0.5*(X-2.0)*(Y+1.0)

ax1.contour(X,Y,Z,levels=18,linewidths=0.8)

ax1.plot(newton_path[:,0],newton_path[:,1],'-o',linewidth=1.4,markersize=4,label='Newton')

ax1.plot(bfgs_path[:,0],bfgs_path[:,1],'-s',linewidth=1.2,markersize=3.5,label='BFGS')

ax1.plot(minimum[0],minimum[1],'r*',markersize=10,label='Minimum')

ax1.set_xlim(-4,4)

ax1.set_ylim(-4,4)

ax1.set_title('Iteration Paths on the Objective Function')

ax1.set_xlabel(r'$x_1$')

ax1.set_ylabel(r'$x_2$')

ax1.grid(True,linestyle=':',alpha=0.20)

ax1.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),ncol=3,frameon=False)

# ============================================================
# OBJECTIVE VALUE VS ITERATION
# ============================================================

newton_values = np.array([f(x) for x in newton_path])

bfgs_values = np.array([f(x) for x in bfgs_path])

ax2.plot(np.arange(len(newton_values)),newton_values,'-o',linewidth=1.3,markersize=4,label='Newton')

ax2.plot(np.arange(len(bfgs_values)),bfgs_values,'-s',linewidth=1.3,markersize=4,label='BFGS')

ax2.set_yscale('log')

ax2.set_title('Objective Function Reduction')

ax2.set_xlabel('Iteration')

ax2.set_ylabel(r'$f(x_k)$')

ax2.grid(True,linestyle=':',alpha=0.25)

ax2.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),ncol=2,frameon=False)

# ============================================================
# LAYOUT
# ============================================================

plt.subplots_adjust(left=0.08,right=0.98,top=0.91,bottom=0.20,wspace=0.28)

# ============================================================
# SINGLE DISPLAY ONLY
# ============================================================

display(fig)

plt.close(fig)